# Oficina de dados 2026.2 — Do que se morre e por que se interna no Brasil (DATASUS)

**Modelagem Informacional · FGV EMAp · Prof. Júlio César Chaves**
Ambiente: **Databricks Free Edition** (serverless, Unity Catalog) · Atividade **individual**

---

### Desafio

> Descobrir, **desde a data mais antiga até a mais recente**, qual a **doença que mais mata** e qual a **doença que mais causa internação hospitalar**, **ao longo dos anos**, em **cada estado brasileiro**.

Fontes: **SIM** (Sistema de Informações sobre Mortalidade) e **SIH/SUS** (Sistema de Informações Hospitalares), do DATASUS, com a **CID-10** como dicionário de doenças.

### A jornada

| # | Etapa | Pergunta que a etapa responde | Saída |
|---|---|---|---|
| 1 | **Find & Understand** | Que dado existe, desde quando, em que grão? O que é "doença" e "estado" nesse dado? | dicionário de dados + 4 decisões de modelagem |
| 2 | **Provision** | Como trago centenas de arquivos `.dbc` para o meu lake, de forma reexecutável? | Volume `bronze_raw` + tabelas `bronze_*` |
| 3 | **Prep** | Como harmonizo anos, códigos e UFs e monto um fato que responda às duas perguntas? | `silver_*`, `dim_cid`, `gold_fato_causa`, `gold_lideres` |
| 4 | **Analyze** | Quem lidera em cada UF e ano? Quando a liderança mudou? A resposta é robusta? | tabela UF × ano + mapa de líderes + leitura temporal |

Cada etapa é uma **linha da Matriz de Zachman** vista na aula 1: Find & Understand ≈ Análise de Requisitos e Modelo Conceitual; Provision ≈ Modelo Físico; Prep ≈ Modelo Lógico e Implementação; Analyze ≈ uso pelo negócio.

### Como este roteiro funciona

- O roteiro entrega **cerca de 70% do caminho pronto** (o *seed*). As células marcadas com **🧩 SUA VEZ** ficam para você: são os outros 30%. Em quase todas, a solução é **adaptar** algo que o seed já fez para o SIM.
- **Python só para trazer os dados; SQL para o resto.** A seção 2 usa Python com Spark para baixar os arquivos e **gerar as tabelas** `bronze_*`. Da seção 3 em diante, toda transformação e análise é **SQL** (células `%sql`), como nas aulas. O Python volta só para desenhar o mapa (4.3) e para chamar o Jev pelo SDK (4.6).
- Comece no **modo `rapido`** (3 UFs pequenas, poucos anos): o pipeline inteiro roda em minutos. Quando tudo funcionar, mude para `completo` e rode de novo. Como cada célula é **idempotente**, só o que falta é baixado.
- Ao fim de cada etapa há uma caixa **Trilhas** com uma atividade por perfil. Faça **pelo menos uma por etapa**:
  **Prático** (executar, quebrar e consertar) · **Criativo** (narrativa, visual, manchete) · **Abstrato** (modelo, propriedade, "e se?").

### Referências

| Fonte | Onde entra |
|---|---|
| FTP DATASUS `ftp.datasus.gov.br/dissemin/publicos/` — `SIM/CID10/DORES`, `SIM/PRELIM/DORES`, `SIHSUS/200801_/dados` | microdados |
| `referencias/datasus/IT_SIHSUS_1603.pdf` — DATASUS, layout da AIH reduzida | colunas do SIH (1.3) |
| `referencias/microdatasus/` — SALDANHA, R., pacote microdatasus (MIT): `process_sim.R` e `process_sih.R` traduzidos para Python (`processa_sim.py`, `processa_sih.py`); livro *sis* (rfsaldanha.github.io/sis/sim.html) | significado dos códigos de cada coluna (1.3) |
| DATASUS — tabelas da **CID-10** (`www2.datasus.gov.br/cid10/V2008`) | dimensão de doenças |
| `referencias/anonimizacao/` — PEREIRA, G. M. C. (TCC EMAp, 2025) | k-anonimato na publicação |
| TypeSafe — primitivas do System One (docs.typesafe.ai/primitives) | Jev: `choice`, `score`, `noul` (4.6) |
| Databricks — função `ai_decide` (docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide) | o mesmo julgamento em SQL (4.7) |

Os artigos e trabalhos citados estão na pasta **`referencias/`** do repositório; os links da web estão em `referencias/README.md`.

## 0. Preparação do ambiente

1. No Databricks Free Edition, traga o repositório da oficina, **`dl-trajetoria`**: **Workspace → Create → Git folder**, cole a URL do repositório e abra `notebooks/oficina_sim_temporal.ipynb`. Se preferir, use **Workspace → Import → File** com este `.ipynb`.
2. Conecte o notebook ao compute **Serverless**.
3. Rode as células de cima para baixo. As duas primeiras instalam as bibliotecas e **reiniciam o Python**. Depois do reinício, as variáveis anteriores se perdem; siga a partir da célula CONFIG.

> **Mensagens esperadas depois do `%pip install`.** Duas mensagens aparecem quase sempre e **não são erro**:
>
> - *"Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython()…"*: a célula seguinte já faz esse reinício.
> - *"Core Python package version(s) changed … idna: 3.11 -> 3.20 …"*: uma dependência das bibliotecas atualizou um pacote que o Databricks já trazia. Em geral nada quebra.
>
> **Se algo quebrar** (erro estranho de import, `display` ou PySpark), restaure o ambiente. No serverless, abra o painel **Environment** (à direita), em **Dependencies → Added** remova o pacote indicado na mensagem e clique em **Apply**. Depois rode de novo a partir da primeira célula. Como o notebook é idempotente, nada é baixado nem duplicado de novo.

> **De onde vêm os dados.** O DATASUS publica por **FTP**, mas esse FTP às vezes fica lento ou fora do ar, e em alguns workspaces do Free Edition o serverless simplesmente não o alcança. Por isso o notebook baixa do **espelho da turma**: os mesmos `.dbc` brutos, intactos, do modo `completo` (coleta de 29/10/2026), publicados por HTTPS na *release* `dados-2026.2` do repositório `dl-trajetoria`. Na célula CONFIG:
>
> - `FONTE = "github"` (padrão): baixa do espelho da turma.
> - `FONTE = "ftp"`: baixa direto do DATASUS; o que falhar é buscado **automaticamente** no espelho. Use se quiser dados mais novos que a coleta do espelho.
>
> Último recurso: rode `python3 scripts/baixar_datasus.py --modo rapido` no seu computador ou no Codespaces e envie a pasta `volume/` gerada para o Volume (**Catalog → workspace → oficina_sus → Volumes → bronze_raw → Upload to this volume**, mantendo as subpastas, ou `databricks fs cp -r volume/ dbfs:/Volumes/workspace/oficina_sus/bronze_raw/`).

> **Custo e cota.** O modo `completo` baixa ~2,5 GB do SIM e ~5 GB do SIH (4 meses por ano). Rode-o **uma vez**, depois de validar tudo no `rapido`.

In [ ]:
%pip install -q pyreaddbc typesafe-sdk

In [ ]:
# Reinicia o Python para carregar os pacotes recém-instalados (fica em célula separada do %pip).
dbutils.library.restartPython()

In [ ]:
# CONFIG ─ parâmetros do roteiro
import os, re, json, tempfile
from pyspark.sql import functions as F

CATALOG = "workspace"        # catálogo padrão do Free Edition
SCHEMA  = "oficina_sus"
VOLUME  = "bronze_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"USE {CATALOG}.{SCHEMA}")   # as células SQL usam nomes curtos: bronze_sim, dim_cid...

def T(nome):
    "Nome totalmente qualificado de uma tabela da oficina."
    return f"{CATALOG}.{SCHEMA}.{nome}"

FTP = "ftp://ftp.datasus.gov.br/dissemin/publicos"
# Espelho da turma: os mesmos .dbc brutos, em pacotes {sistema}_{UF}.tar, por HTTPS
ESPELHO = "https://github.com/Escola-de-Matematica-Aplicada/dl-trajetoria/releases/download/dados-2026.2"
FONTE = "github"  # "github": espelho da turma · "ftp": direto do DATASUS, com o espelho para o que falhar

# Código IBGE da UF (2 primeiros dígitos do código de município) → sigla
UF_COD = {"11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
          "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
          "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR", "42": "SC", "43": "RS",
          "50": "MS", "51": "MT", "52": "GO", "53": "DF"}

MODO = "rapido"   # "rapido": valida o pipeline em minutos · "completo": responde ao desafio
if MODO == "rapido":
    UFS            = ["AC", "AP", "RR"]
    ANOS_SIM       = [1996, 2008, 2019, 2020, 2021, 2024]
    ANOS_SIM_PRELIM = []
    ANOS_SIH       = [2008, 2019, 2020, 2021, 2025]
    MESES_SIH      = [1, 7]
else:
    UFS            = sorted(UF_COD.values())
    ANOS_SIM       = list(range(1996, 2025))   # definitivos (ver seção 1)
    ANOS_SIM_PRELIM = [2025]                   # preliminar: ano fechado mais recente
    ANOS_SIH       = list(range(2008, 2026))
    MESES_SIH      = [1, 4, 7, 10]             # 1 mês por trimestre (ver seção 4.5)
print(f"MODO={MODO} · {len(UFS)} UFs · lake bruto em {RAW}")

---
# 1. Find & Understand

## 1.1 Decompor a pergunta antes de procurar o dado

Cada palavra do desafio esconde uma **decisão de modelagem**. Responda primeiro no papel:

| Palavra do desafio | Pergunta que ela levanta | Onde decidimos |
|---|---|---|
| "doença" | Em que nível da CID-10? `I219` (subcategoria), `I21` infarto (categoria), `I20–I25` isquêmicas (grupo) ou capítulo IX circulatório? Parto é doença? Homicídio é doença? | 3.4 e 4.5 |
| "mais mata" | Que campo da declaração de óbito? Ela tem até 5 linhas de causas. | 1.2 (`CAUSABAS`, a **causa básica**) |
| "mais causa internação" | Contar AIH, pessoas ou dias? Toda AIH é uma internação nova? | 1.3 e 3.2 |
| "cada estado" | Estado onde a pessoa **mora** ou onde **morreu/foi internada**? | 3.2 |
| "ao longo dos anos" / "mais antiga até a mais recente" | Desde quando cada sistema existe no mesmo formato? Qual é o último ano **fechado**? | 1.2 |

## 1.2 Encontrar as fontes e a sua cobertura temporal

| Sistema | Pasta no FTP | Período | Classificação | Veredito |
|---|---|---|---|---|
| SIM (CID-9) | `SIM/CID9/DORES` | 1979–1995 | **CID-9** | ⚠️ Fora: outra classificação de doenças. Juntar as duas exige uma tabela de correspondência (trilha abstrata). |
| SIM | `SIM/CID10/DORES` | **1996 → 2024** | CID-10 | ✅ Dado **definitivo**, um arquivo por UF de **residência** e ano: `DO{UF}{AAAA}.dbc`. |
| SIM preliminar | `SIM/PRELIM/DORES` | 2025 → 2026 | CID-10 | ⚠️ Ainda muda. 2025 entra marcado como `preliminar`; 2026 está incompleto e fica fora. |
| SIH (layout antigo) | `SIHSUS/199201_200712` | 1992–2007 | CID-9 até 1997 | ⚠️ Fora: layout e classificação diferentes. |
| SIH | `SIHSUS/200801_/dados` | **2008-01 → mês mais recente** | CID-10 | ✅ AIH "reduzida", um arquivo por UF **do hospital** e **mês de competência**: `RD{UF}{AA}{MM}.dbc`. |
| CID-10 | `www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip` | versão 2008 | — | ✅ Categorias, grupos e capítulos. |

Os arquivos são **`.dbc`**: um DBF (formato dBase, de largura fixa) comprimido com um algoritmo próprio do DATASUS. Usaremos `pyreaddbc` para descomprimir e um leitor de DBF escrito aqui mesmo (seção 2). O caminho de linha de comando citado no syllabus (`dbf_dump`) só funciona **depois** de descomprimir.

### Checkpoint 1: entender antes de baixar

A célula abaixo **não baixa microdados**: só lista as pastas do FTP e calcula a cobertura.

In [ ]:
from ftplib import FTP as ConexaoFTP

def listar(pasta):
    with ConexaoFTP("ftp.datasus.gov.br", timeout=60) as ftp:
        ftp.login()
        return [os.path.basename(n) for n in ftp.nlst(f"/dissemin/publicos/{pasta}")]

try:
    sim = [n for n in listar("SIM/CID10/DORES") if re.match(r"DO[A-Z]{2}\d{4}\.dbc$", n, re.I)]
    anos = sorted({int(n[4:8]) for n in sim})
    print(f"SIM definitivo: {anos[0]} → {anos[-1]} · {len(sim)} arquivos")
    print("  'UFs' nos nomes:", sorted({n[2:4].upper() for n in sim}))
    prelim = [n for n in listar("SIM/PRELIM/DORES") if n.upper().startswith("DO")]
    print("SIM preliminar:", sorted({n[4:8] for n in prelim}))
    rd = [n for n in listar("SIHSUS/200801_/dados") if re.match(r"RD[A-Z]{2}\d{4}\.dbc$", n, re.I)]
    comp = sorted({"20" + n[4:8] for n in rd})
    print(f"SIH (RD): {comp[0][:4]}-{comp[0][4:]} → {comp[-1][:4]}-{comp[-1][4:]} · {len(rd)} arquivos")
except Exception as e:
    print("Falha ao acessar o FTP do DATASUS (instável? tente de novo em alguns minutos):", e)
    print('A seção 2 não depende disso: com FONTE = "github" (padrão) ela baixa do espelho da turma.')
    print("No Codespaces/terminal: curl --list-only ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/")
    print("No navegador (que não abre ftp://): https://datasus.saude.gov.br/transferencia-de-arquivos/")

> **Observe:** há **28** "UFs" nos nomes do SIM, não 27. A sobra é `BR`: o arquivo `DOBR{AAAA}.dbc` traz **o Brasil inteiro**, a mesma informação da soma das UFs. Lê-lo junto com as UFs **duplica** os óbitos. É a mesma armadilha do `perfil_eleitorado_BRASIL.csv` do TSE.
>
> Na pasta do SIH também há arquivos `RJ`, `SP` e `ER` (AIH rejeitadas, serviços profissionais, erros). Só o prefixo **`RD`** é a AIH reduzida. Cuidado: `RJ` aqui é tipo de arquivo, não o estado!

## 1.3 Dicionário de dados

Duas fontes, ambas na pasta `referencias/` do repositório:

- **Layout oficial** do DATASUS: `referencias/datasus/IT_SIHSUS_1603.pdf` (no FTP, em `SIHSUS/200801_/Doc/`). Lista **todas** as colunas da AIH reduzida, com tipo e descrição.
- **microdatasus** (SALDANHA, R.) em Python: `referencias/microdatasus/processa_sim.py` e `processa_sih.py`, traduzidos do pacote R original. O dicionário `CODIGOS` mostra o que cada código significa (ex.: `CODIGOS["SEXO"]` → `{'1': 'Masculino', ...}`). Abra o arquivo e leia, ou importe-o numa célula:

  ```python
  import sys; sys.path.append("../referencias/microdatasus")
  from processa_sih import CODIGOS
  CODIGOS["SEXO"]
  ```

  Colunas que o pacote não recodifica, como os diagnósticos, não estão em `CODIGOS`; procure-as no PDF. Há também `processar(df)`, que decodifica um `pandas.DataFrame` inteiro. É opcional e serve para explorar uma amostra.

**SIM** (preenchido pelo seed):

| Atributo do desafio | Coluna | Formato | Observação |
|---|---|---|---|
| doença que matou | `CAUSABAS` | CID-10 sem ponto, 3 ou 4 caracteres (`I219`, `I10X` ou `I10`) | **causa básica**: a doença que iniciou a cadeia de eventos |
| ano | nome do arquivo (`DO{UF}{AAAA}`) e `DTOBITO` (`ddmmaaaa`) | texto | |
| estado | `CODMUNRES` | município IBGE de residência: **7 dígitos até 2005, 6 depois** | os 2 primeiros dígitos são a UF |
| contexto | `SEXO`, `IDADE` | `IDADE` = unidade (1º dígito) + valor | não usados no desafio; úteis na trilha |

**SIH** — 🧩 **SUA VEZ (TODO 1.1):** preencha a tabela abaixo, no mesmo formato da tabela do SIM. Primeiro ache cada coluna no PDF `referencias/datasus/IT_SIHSUS_1603.pdf`: a de diagnóstico, a de ano (e mês) de competência, a de município de residência e a que identifica o **tipo da AIH**. Depois abra `referencias/microdatasus/processa_sih.py`, procure `'IDENT'` em `CODIGOS` e anote o significado de cada código. Você vai precisar dele na etapa 3.

| Atributo do desafio | Coluna | Formato | Observação |
|---|---|---|---|
| doença que causou a internação | | | |
| ano | | | |
| estado | | | |
| tipo de AIH | | | |

---
### Trilhas — Etapa 1

- **Prático:** baixe `DOAC2022.dbc` (óbitos do Acre em 2022) e abra o PDF `Estrutura_do_SIM_2025.pdf`. Confira o significado de `CAUSABAS` e de `CIRCOBITO`.
  - **PDF:** já está no repositório, em `referencias/datasus/Estrutura_do_SIM_2025.pdf`. No FTP fica em `ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DOCS/Estrutura_do_SIM_2025.pdf`.
  - **`.dbc` no próprio Databricks** (se o FTP responder daqui; a seção 2 não depende dele). Crie uma célula com:

    ```python
    from ftplib import FTP as ConexaoFTP
    destino = os.path.join(tempfile.gettempdir(), "DOAC2022.dbc")     # disco local do notebook
    with ConexaoFTP("ftp.datasus.gov.br", timeout=60) as ftp, open(destino, "wb") as f:
        ftp.login()
        ftp.retrbinary("RETR /dissemin/publicos/SIM/CID10/DORES/DOAC2022.dbc", f.write)
    print(destino, os.path.getsize(destino), "bytes")
    ```

    Na etapa 2, `pyreaddbc.dbc2dbf` e `ler_dbf` transformam esse arquivo em tabela.
  - **`.dbc` pelo navegador:** Chrome e Firefox não abrem mais links `ftp://`. Use a página do DATASUS **https://datasus.saude.gov.br/transferencia-de-arquivos/**: preencha **Fonte** = SIM, **Modalidade** = Dados, **Tipo de Arquivo** = DO (declarações de óbito), **Ano** = 2022 e **UF** = AC, e baixe o arquivo listado.
  - **`.dbc` pelo terminal** (Codespaces ou seu computador): `curl -O ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2022.dbc`.
  - Para abrir o `.dbc` fora do Databricks: `pip install pyreaddbc` e `pyreaddbc.dbc2dbf("DOAC2022.dbc", "DOAC2022.dbf")`. O DBF abre no LibreOffice Calc ou com a função `ler_dbf` da etapa 2.
- **Criativo:** escreva, em 3 linhas, a "persona" de um gestor da Secretaria Estadual de Saúde que usaria esta tabela. Que decisão de orçamento ele tomaria com ela?
- **Abstrato:** o SIM organiza os arquivos por UF de **residência** e o SIH por UF **do hospital**. Em qual coluna de Zachman (o quê, como, onde, quem, quando, por quê) mora essa diferença? Que pergunta de negócio cada escolha responde melhor?

---
# 2. Provision (camada bronze)

Princípio: **o bronze guarda o dado como veio**, mais metadados de linhagem. Aqui o arquivo `.dbc` original fica intacto no Volume. A tabela bronze leva **só as colunas que a pergunta usa**, ainda como texto e sem nenhuma limpeza. Se precisarmos de outra coluna depois, reprocessamos a partir do `.dbc`.

## 2.1 Baixar de forma idempotente

Uma operação é **idempotente** quando rodá-la 1 ou N vezes deixa o lake no mesmo estado: f(f(x)) = f(x). É o que permite reexecutar uma célula que falhou sem pensar no que já rodou. No Spark, `overwrite` e `MERGE INTO` são idempotentes; `append` e `INSERT INTO` não são, porque cada execução duplica as linhas.

A função abaixo pula o arquivo que já está no Volume. Mas "já está" só é verdade se o arquivo estiver **completo**. Por isso a cópia vai primeiro para `destino.part` e só no fim é **renomeada** para `destino`. Se a cópia cair no meio, sobra um `.part` (apagado no `finally`), e a próxima execução baixa o arquivo de novo.

In [ ]:
import shutil, urllib.request

def abrir(url):
    req = urllib.request.Request(url, headers={"User-Agent": "oficina-emap/2026"})
    return urllib.request.urlopen(req, timeout=300)

def gravar(fluxo, destino):
    "Copia `fluxo` para `destino` (caminho no Volume) sem nunca deixar `destino` pela metade."
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    tmp = os.path.join(tempfile.gettempdir(), os.path.basename(destino) + ".part")
    parcial = destino + ".part"
    try:
        with open(tmp, "wb") as f:
            shutil.copyfileobj(fluxo, f, 1 << 20)
        shutil.copyfile(tmp, parcial)       # baixa no disco local, depois copia para o Volume
        os.replace(parcial, destino)        # renomear é atômico: `destino` nunca fica pela metade
    finally:
        for p in (tmp, parcial):
            if os.path.exists(p):
                os.remove(p)

def baixar(url, destino):
    "Baixa `url` para `destino`. Não baixa de novo se já existe."
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return "já no lake"
    try:
        with abrir(url) as r:
            gravar(r, destino)
        return "baixado"
    except Exception as e:
        return f"FALHOU ({e})"

## 2.2 Ler DBF: um formato de **largura fixa**

Um DBF é um cabeçalho (nome, tipo e largura de cada campo) seguido de registros **todos do mesmo tamanho**, um atrás do outro. É um formato **orientado a linha**: para ler uma coluna, você precisa passar por todas as linhas. Por isso o convertemos para **Parquet**, que é **colunar**.

A largura fixa tem uma vantagem: o arquivo inteiro vira uma matriz de bytes (`numpy`), e cada coluna é só uma fatia dela. Sem laços em Python, um ano inteiro de SP é lido em segundos.

In [ ]:
import struct, numpy as np, pandas as pd, pyreaddbc

def ler_dbf(caminho, colunas):
    "Lê só `colunas` de um DBF como texto. Colunas ausentes (layout antigo) voltam vazias."
    with open(caminho, "rb") as f:
        n_reg, tam_cab, tam_reg = struct.unpack("<IHH", f.read(32)[4:12])
        campos, pos = {}, 1                       # byte 0 de cada registro = marca de apagado
        while (d := f.read(32))[0] != 0x0D:       # descritores de 32 bytes até o terminador 0x0D
            nome = d[:11].split(b"\0")[0].decode("latin-1").upper()
            campos[nome] = (pos, d[16])           # (posição, largura)
            pos += d[16]
        f.seek(tam_cab)
        bruto = np.frombuffer(f.read(n_reg * tam_reg), dtype=np.uint8)
    bruto = bruto[: len(bruto) // tam_reg * tam_reg].reshape(-1, tam_reg)
    bruto = bruto[bruto[:, 0] != ord("*")]        # descarta registros apagados
    df = {}
    for c in colunas:
        if c in campos:
            p, w = campos[c]
            fatia = np.ascontiguousarray(bruto[:, p:p + w]).view(f"S{w}").ravel()
            df[c] = pd.Series(fatia).str.decode("latin-1").str.strip()
        else:
            df[c] = pd.Series([None] * len(bruto), dtype="string")
    return pd.DataFrame(df).astype("string")

def dbc_para_parquet(sistema, nome_dbc, colunas):
    "Volume: dbc/{sistema}/X.dbc  →  parquet/{sistema}/X.parquet (só `colunas`)."
    origem  = f"{RAW}/dbc/{sistema}/{nome_dbc}"
    destino = f"{RAW}/parquet/{sistema}/{nome_dbc[:-4]}.parquet"
    if os.path.exists(destino):
        return
    local = os.path.join(tempfile.gettempdir(), nome_dbc[:-4])
    try:
        pyreaddbc.dbc2dbf(origem, local + ".dbf")
        ler_dbf(local + ".dbf", colunas).to_parquet(local + ".parquet", index=False)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        shutil.copyfile(local + ".parquet", destino)
    finally:
        for ext in (".dbf", ".parquet"):
            if os.path.exists(local + ext):
                os.remove(local + ext)

## 2.3 Orquestrar: baixar + converter, em paralelo

`provisionar` recebe uma lista de nomes de arquivo e faz as duas etapas para cada um. Com `FONTE = "github"` (padrão), os `.dbc` vêm do **espelho da turma** (`do_espelho`); com `"ftp"`, o espelho cobre o que o FTP não entregar. O espelho tem um pacote `.tar` por sistema e UF, com os `.dbc` brutos dentro, lido em fluxo, sem guardar o `.tar`. Ao final, mostra o **Plano B** para o que ainda faltar.

In [ ]:
import tarfile
from concurrent.futures import ThreadPoolExecutor

def do_espelho(sistema, nomes, paralelo=4):
    "Extrai `nomes` dos pacotes {sistema}_{UF}.tar do ESPELHO direto para dbc/{sistema}/ no Volume."
    faltam = {n for n in nomes if not os.path.exists(f"{RAW}/dbc/{sistema}/{n}")}
    def uma_uf(uf):
        try:
            with abrir(f"{ESPELHO}/{sistema}_{uf}.tar") as r, tarfile.open(fileobj=r, mode="r|") as tar:
                for m in tar:                     # "r|" = fluxo: lê o .tar uma vez, do início ao fim
                    nome = os.path.basename(m.name)
                    if nome in faltam:
                        gravar(tar.extractfile(m), f"{RAW}/dbc/{sistema}/{nome}")
        except Exception as e:
            print(f"  ✗ espelho {sistema}_{uf}.tar: {e}")
    with ThreadPoolExecutor(paralelo) as ex:
        list(ex.map(uma_uf, sorted({n[2:4] for n in faltam})))

def provisionar(sistema, pasta_ftp, nomes, colunas, paralelo=4):
    if FONTE == "github":
        do_espelho(sistema, nomes, paralelo)      # traz os .dbc; abaixo só converte ("já no lake")
    def um(nome):
        st = baixar(f"{FTP}/{pasta_ftp}/{nome}", f"{RAW}/dbc/{sistema}/{nome}")
        if not st.startswith("FALHOU"):
            dbc_para_parquet(sistema, nome, colunas)
        return nome, st
    with ThreadPoolExecutor(paralelo) as ex:
        res = list(ex.map(um, nomes))
    falhas = [n for n, s in res if s.startswith("FALHOU")]
    if falhas and FONTE == "ftp":
        print(f"{sistema}: {len(falhas)} arquivo(s) falharam no FTP → buscando no espelho da turma")
        do_espelho(sistema, falhas, paralelo)
        with ThreadPoolExecutor(paralelo) as ex:
            res = [r for r in res if r[0] not in falhas] + list(ex.map(um, falhas))
    falhas = [(n, s) for n, s in res if s.startswith("FALHOU")]
    print(f"{sistema}: {len(res) - len(falhas)}/{len(res)} arquivos no lake")
    for n, s in falhas[:10]:
        print(f"  ✗ {n}: {s}")
    if falhas:
        print(f"  PLANO B → python3 scripts/baixar_datasus.py e envie para {RAW}/dbc/{sistema}/ ; depois rode de novo.")
    return res

COLUNAS_SIM = ["DTOBITO", "CAUSABAS", "CODMUNRES", "SEXO", "IDADE"]
NOMES_SIM   = [f"DO{uf}{ano}.dbc" for uf in UFS for ano in ANOS_SIM]          # nunca DOBR!
_ = provisionar("sim", "SIM/CID10/DORES", NOMES_SIM, COLUNAS_SIM)
if ANOS_SIM_PRELIM:
    _ = provisionar("sim_prelim", "SIM/PRELIM/DORES",
                    [f"DO{uf}{ano}.dbc" for uf in UFS for ano in ANOS_SIM_PRELIM], COLUNAS_SIM)

In [ ]:
# 🧩 SUA VEZ (TODO 2.1) — provisionar o SIH
# Os arquivos seguem o padrão RD{UF}{AA}{MM}.dbc, ex.: RDAC2401.dbc = Acre, jan/2024.
# Use UFS, ANOS_SIH e MESES_SIH. Atenção ao ano com 2 dígitos.
NOMES_SIH   = ...   # lista de nomes de arquivo
COLUNAS_SIH = ...   # as colunas do seu dicionário (TODO 1.1); inclua também "MES_CMPT"

if NOMES_SIH is Ellipsis or COLUNAS_SIH is Ellipsis:
    print("⚠️  TODO 2.1 ainda não resolvido: o SIH não foi provisionado. O notebook segue só com o SIM.")
else:
    _ = provisionar("sih", "SIHSUS/200801_/dados", NOMES_SIH, COLUNAS_SIH)

## 2.4 CID-10 e tabelas bronze

In [ ]:
import zipfile

origens = ["http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip", f"{ESPELHO}/CID10CSV.zip"]
for url in (origens if FONTE == "ftp" else origens[::-1]):
    st = baixar(url, f"{RAW}/cid/CID10CSV.zip")
    if not st.startswith("FALHOU"):
        break
print("CID10CSV.zip:", st, "·", url)
with zipfile.ZipFile(f"{RAW}/cid/CID10CSV.zip") as z:
    for arq, tabela in [("CID-10-CAPITULOS.CSV", "bronze_cid_capitulos"),
                        ("CID-10-GRUPOS.CSV", "bronze_cid_grupos"),
                        ("CID-10-CATEGORIAS.CSV", "bronze_cid_categorias")]:
        cid = pd.read_csv(z.open(arq), sep=";", encoding="latin-1", dtype=str)
        cid = cid.loc[:, ~cid.columns.str.startswith("Unnamed")]       # ';' no fim de cada linha
        spark.createDataFrame(cid.fillna("")).write.mode("overwrite") \
             .option("overwriteSchema", True).saveAsTable(T(tabela))

def bronze(sistema, tabela, colunas):
    pasta = f"{RAW}/parquet/{sistema}"
    if not os.path.isdir(pasta) or not os.listdir(pasta):
        # tabela VAZIA com as colunas certas: as consultas SQL das seções seguintes rodam mesmo assim
        esquema = ", ".join(f"{c} STRING" for c in colunas) + ", _arquivo STRING, _ingestao TIMESTAMP"
        spark.createDataFrame([], esquema.lstrip(", ")).write.mode("overwrite") \
             .option("overwriteSchema", True).saveAsTable(T(tabela))
        print(f"– {tabela}: nada em {pasta} (tabela vazia)")
        return
    (spark.read.option("mergeSchema", True).parquet(pasta)
          .withColumn("_arquivo", F.col("_metadata.file_name"))
          .withColumn("_ingestao", F.current_timestamp())
          .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T(tabela)))
    print(f"✓ {tabela}: {spark.table(T(tabela)).count():,} linhas")

bronze("sim", "bronze_sim", COLUNAS_SIM)
bronze("sim_prelim", "bronze_sim_prelim", COLUNAS_SIM)
bronze("sih", "bronze_sih", [] if COLUNAS_SIH is Ellipsis else COLUNAS_SIH)

---
### Trilhas — Etapa 2

- **Prático:** rode de novo todas as células da seção 2 e confirme que nada é baixado outra vez. Depois apague um `.parquet` do Volume e rode de novo: só ele é refeito.
- **Criativo:** desenhe o "mapa do lake" (FTP → `dbc/` → `parquet/` → `bronze_*` → `silver_*` → `gold_*`) como se fosse outro lugar onde coisas chegam, são guardadas e transformadas. Por exemplo, um **porto marítimo**: o navio (FTP) traz **contêineres lacrados** (`.dbc`), que são descarregados no cais (`dbc/`), abertos e reorganizados em paletes (`parquet/`), registrados na alfândega como chegaram (`bronze_*`), conferidos e separados por destino (`silver_*`) e despachados prontos para a loja (`gold_*`). Outras ideias: uma **cozinha** (feira → despensa → pré-preparo → prato) ou os **Correios** (coleta → triagem → centro de distribuição → entrega). O que **não** cabe na sua metáfora também é achado: onde ela quebra?
- **Abstrato:** compare o custo de ler **uma coluna** de 1 milhão de óbitos em DBF (linha, largura fixa) e em Parquet (colunar, comprimido). Por que o leitor `ler_dbf` acima consegue ser rápido mesmo sendo um formato de linha? Que propriedade do formato ele explora?

---
# 3. Prep (camadas silver e gold)

Daqui em diante, **tudo é SQL**. O Python da seção 2 já fez o trabalho dele: as tabelas `bronze_*` existem no schema `oficina_sus`. Cada célula começa com `%sql` e termina, quase sempre, com um `SELECT` que mostra o resultado. As células usam nomes curtos (`bronze_sim`, não `workspace.oficina_sus.bronze_sim`) porque a CONFIG fez `USE workspace.oficina_sus`. Se a sessão reiniciar, rode a CONFIG de novo.

## 3.1 Conhecer o bronze antes de transformar

In [ ]:
%sql
USE workspace.oficina_sus;

SELECT substring(_arquivo, 5, 4)                                  AS ano_arquivo,
       count(*)                                                   AS obitos,
       round(avg(length(CODMUNRES)), 2)                           AS digitos_CODMUNRES,
       count_if(length(CAUSABAS) = 3)                             AS causabas_3_chars,
       count_if(CAUSABAS LIKE '%X')                               AS causabas_terminada_em_X,
       count_if(substring(DTOBITO, 5, 4) <> substring(_arquivo, 5, 4)) AS ano_dtobito_difere
FROM bronze_sim
GROUP BY substring(_arquivo, 5, 4)
ORDER BY ano_arquivo

> **Leitura.** O código de município muda de 7 para 6 dígitos ao longo da série, e uma mesma categoria aparece como `I10`, `I10X` ou `I109`. Os **3 primeiros caracteres** da CID e os **2 primeiros** do município são estáveis: é neles que harmonizamos.

## 3.2 Silver: um registro por óbito e por internação, com chaves harmonizadas

Decisões do seed:
- **estado = UF de residência** (2 primeiros dígitos do município). O código IBGE vira sigla por uma dimensão pequena, `dim_uf`. Código ignorado → `uf` nula, contada e excluída.
- **doença = categoria CID-10** (3 caracteres). Revisitamos isso na seção 4.5.
- **ano do óbito = ano do arquivo** (o SIM organiza os arquivos por ano do óbito).

In [ ]:
%sql
-- Código IBGE da UF (2 primeiros dígitos do código de município) → sigla
CREATE OR REPLACE TABLE dim_uf AS
SELECT * FROM VALUES
  ('11', 'RO'), ('12', 'AC'), ('13', 'AM'), ('14', 'RR'), ('15', 'PA'), ('16', 'AP'), ('17', 'TO'),
  ('21', 'MA'), ('22', 'PI'), ('23', 'CE'), ('24', 'RN'), ('25', 'PB'), ('26', 'PE'), ('27', 'AL'), ('28', 'SE'), ('29', 'BA'),
  ('31', 'MG'), ('32', 'ES'), ('33', 'RJ'), ('35', 'SP'), ('41', 'PR'), ('42', 'SC'), ('43', 'RS'),
  ('50', 'MS'), ('51', 'MT'), ('52', 'GO'), ('53', 'DF')
AS t(codigo, uf);

CREATE OR REPLACE TABLE silver_obitos AS
SELECT CAST(substring(b._arquivo, 5, 4) AS INT) AS ano,
       u.uf,
       upper(substring(b.CAUSABAS, 1, 3))       AS cid_cat,
       b.preliminar,
       b._arquivo
FROM (SELECT _arquivo, CODMUNRES, CAUSABAS, false AS preliminar FROM bronze_sim
      UNION ALL
      SELECT _arquivo, CODMUNRES, CAUSABAS, true  AS preliminar FROM bronze_sim_prelim) b
LEFT JOIN dim_uf u ON u.codigo = substring(b.CODMUNRES, 1, 2);

SELECT preliminar, count(*) AS obitos, count_if(uf IS NULL) AS uf_ignorada, min(ano) AS de, max(ano) AS ate
FROM silver_obitos
GROUP BY preliminar

In [ ]:
%sql
-- 🧩 SUA VEZ (TODO 3.1) — silver_internacoes
-- Mesmo formato de silver_obitos: ano, uf (de RESIDÊNCIA), cid_cat, mes, _arquivo.
-- Lembre do seu dicionário (TODO 1.1): qual valor de IDENT é uma internação NOVA?
-- Como está, a célula cria a tabela VAZIA com o formato certo, e o notebook roda até o fim.
-- Troque cada NULL pela expressão certa, a última linha por FROM bronze_sih ... e acrescente o filtro do tipo de AIH.
CREATE OR REPLACE TABLE silver_internacoes AS
SELECT CAST(NULL AS INT)    AS ano,       -- 🧩 ano de competência
       CAST(NULL AS STRING) AS uf,        -- 🧩 UF de residência (use dim_uf, como no SIM)
       CAST(NULL AS STRING) AS cid_cat,   -- 🧩 categoria CID-10 (3 caracteres) do diagnóstico principal
       CAST(NULL AS INT)    AS mes,       -- 🧩 mês de competência
       CAST(NULL AS STRING) AS _arquivo
LIMIT 0;

SELECT ano, count(*) AS internacoes FROM silver_internacoes GROUP BY ano ORDER BY ano

## 3.3 Dimensão de doenças (`dim_cid`)

A CID-10 tem uma hierarquia **capítulo → grupo → categoria**. Os capítulos e grupos vêm como **intervalos** de categorias (`A00`–`B99`), então a junção é por faixa (`BETWEEN`), não por igualdade. A comparação de texto funciona porque todos os códigos têm o formato letra + 2 dígitos.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cid_juncao AS
SELECT cat.CAT                  AS cid_cat,
       cat.DESCRICAO            AS descricao,
       CAST(cap.NUMCAP AS INT)  AS capitulo,
       cap.DESCRABREV           AS capitulo_desc,
       grp.CATINIC              AS g_ini,
       grp.CATFIM               AS g_fim,
       grp.DESCRICAO            AS grupo_desc
FROM bronze_cid_categorias cat
LEFT JOIN bronze_cid_capitulos cap ON cat.CAT BETWEEN cap.CATINIC AND cap.CATFIM
LEFT JOIN bronze_cid_grupos    grp ON cat.CAT BETWEEN grp.CATINIC AND grp.CATFIM;

SELECT (SELECT count(*) FROM bronze_cid_categorias) AS categorias,
       count(*)                                     AS linhas_depois_da_juncao_por_faixa
FROM cid_juncao

In [ ]:
%sql
-- Quais categorias aparecem mais de uma vez depois da junção?
SELECT cid_cat, g_ini, g_fim, grupo_desc
FROM cid_juncao
WHERE cid_cat IN (SELECT cid_cat FROM cid_juncao GROUP BY cid_cat HAVING count(*) > 1)
ORDER BY cid_cat, g_ini
LIMIT 9

> **Armadilha: a junção por faixa duplicou linhas.** Os grupos de neoplasias são **aninhados**: `C00–C97` (neoplasias malignas) contém `C00–C75`, que contém `C00–C14` (lábio, cavidade oral e faringe). A tabela de grupos é, na verdade, uma **hierarquia**, e não uma partição. Decisão do seed: cada categoria fica com o grupo **mais específico**, o de início mais tardio e fim mais cedo (`ROW_NUMBER()` abaixo).

> **Armadilha semântica: COVID-19.** No Brasil, a COVID-19 foi codificada como **`B34.2`** ("infecção por coronavírus de localização não especificada"), e não como `U07.1`. Na categoria de 3 caracteres ela aparece como **`B34` — "Doença por vírus, de localização não especificada"**. A descrição oficial esconde a doença que mais matou em 2020–2021. Nenhuma regra sobre o texto da CID descobriria isso: é **conhecimento de domínio** que precisa entrar no modelo. Por isso a `dim_cid` ganha um atributo `observacao`, vindo de uma tabela pequena escrita à mão.

In [ ]:
%sql
CREATE OR REPLACE TABLE cid_observacoes AS
SELECT * FROM VALUES
  ('B34', 'Em 2020–2022 dominada por B34.2 = COVID-19 (codificação adotada no Brasil)')
AS t(cid_cat, observacao);

CREATE OR REPLACE TABLE dim_cid AS
SELECT j.cid_cat, j.descricao, j.capitulo, j.capitulo_desc,
       concat_ws('-', j.g_ini, j.g_fim) AS grupo, j.grupo_desc, o.observacao
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY cid_cat ORDER BY g_ini DESC NULLS LAST, g_fim ASC) AS r
      FROM cid_juncao) j
LEFT JOIN cid_observacoes o ON o.cid_cat = j.cid_cat
WHERE j.r = 1;

SELECT count(*) AS linhas, count(DISTINCT cid_cat) AS distintas FROM dim_cid

In [ ]:
%sql
-- Códigos usados nos microdados que NÃO estão na tabela da CID
SELECT u.cid_cat, count(*) AS eventos
FROM (SELECT cid_cat FROM silver_obitos UNION ALL SELECT cid_cat FROM silver_internacoes) u
LEFT ANTI JOIN dim_cid d ON d.cid_cat = u.cid_cat
GROUP BY u.cid_cat
ORDER BY eventos DESC

> **Confira:** agora "linhas" e "distintas" são iguais. Se aparecer algum código órfão acima (sem descrição na CID de 2008, como códigos `U` criados depois), ele fica fora da `dim_cid` e com `capitulo` nulo no fato: documente-o.

## 3.4 O que conta como "doença"? (decisão de modelagem)

Sem filtro, a "doença" que mais interna no Brasil é o **parto normal** (`O80`), e entre as que mais matam aparece **`R99` — "outras causas mal definidas"**. Nenhuma das duas responde ao espírito da pergunta. O seed exclui cinco capítulos.

A decisão vira **dado**: a tabela `capitulos_fora`, com o motivo de cada exclusão. Em vez de apagar essas linhas, guardamos tudo no gold e **filtramos na consulta** (`LEFT ANTI JOIN capitulos_fora`). Assim a decisão pode ser revista com um `INSERT` ou `DELETE` nessa tabela, sem reprocessar nada.

In [ ]:
%sql
CREATE OR REPLACE TABLE capitulos_fora AS
SELECT * FROM VALUES
  (15, 'Gravidez, parto e puerpério (O)'),
  (18, 'Sintomas e achados anormais: causas mal definidas (R)'),
  (19, 'Lesões e envenenamentos (S–T): a natureza da lesão, não uma doença'),
  (20, 'Causas externas (V–Y): acidentes, agressões, suicídios'),
  (21, 'Fatores que influenciam o contato com serviços de saúde (Z)')
AS t(capitulo, motivo);

SELECT d.capitulo, d.capitulo_desc, f.motivo, count(*) AS categorias
FROM dim_cid d
JOIN capitulos_fora f ON f.capitulo = d.capitulo
GROUP BY d.capitulo, d.capitulo_desc, f.motivo
ORDER BY d.capitulo

## 3.5 Gold: um fato para as duas perguntas

Grão: **quantidade de eventos de uma fonte (óbito ou internação), em um ano, em uma UF de residência, por categoria CID**. As duas perguntas do desafio viram **dois valores de `fonte`** na mesma tabela, como o formato longo do roteiro do IPG.

Só entram as UFs **cujos arquivos foram carregados** naquela fonte (a sigla está no nome do arquivo: `DO`**`AC`**`2019`, `RD`**`AC`**`1901`). No modo `rapido`, os arquivos do SIH do Acre trazem também pacientes que **moram** no Amazonas. Mas os amazonenses internados em Manaus estão em `RDAM…`, que não baixamos. Contar o AM a partir dos arquivos do AC daria um número parcial e enganoso.

In [ ]:
%sql
-- 🧩 SUA VEZ (TODO 3.2) — juntar as duas fontes no mesmo fato
CREATE OR REPLACE TABLE gold_fato_causa AS
WITH eventos AS (
  SELECT 'obito' AS fonte, ano, uf, cid_cat, _arquivo FROM silver_obitos
  -- 🧩 ... e as internações? (UNION ALL, com fonte = 'internacao')
),
carregadas AS (SELECT DISTINCT fonte, substring(_arquivo, 3, 2) AS uf FROM eventos)
SELECT e.fonte, e.ano, e.uf, e.cid_cat, d.capitulo, count(*) AS n
FROM eventos e
JOIN carregadas c      ON c.fonte = e.fonte AND c.uf = e.uf   -- só UFs com arquivos carregados
LEFT JOIN dim_cid d    ON d.cid_cat = e.cid_cat
GROUP BY e.fonte, e.ano, e.uf, e.cid_cat, d.capitulo;

SELECT fonte, sum(n) AS eventos, min(ano) AS de, max(ano) AS ate FROM gold_fato_causa GROUP BY fonte

### Ranking: a doença líder em cada UF e ano

`gold_lideres` aplica a decisão do 3.4 (`LEFT ANTI JOIN capitulos_fora`) e numera as causas por fonte, UF e ano com uma **função de janela** (`ROW_NUMBER() OVER (PARTITION BY …)`). Guardamos as 3 primeiras de cada grupo.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_lideres AS
WITH soma AS (
  SELECT g.fonte, g.ano, g.uf, g.cid_cat, sum(g.n) AS n
  FROM gold_fato_causa g
  LEFT ANTI JOIN capitulos_fora f ON f.capitulo = g.capitulo     -- capitulo nulo nunca casa: fica
  GROUP BY g.fonte, g.ano, g.uf, g.cid_cat
),
ranking AS (
  SELECT *,
         round(n / sum(n) OVER (PARTITION BY fonte, ano, uf), 4)                AS pct,
         ROW_NUMBER() OVER (PARTITION BY fonte, ano, uf ORDER BY n DESC, cid_cat) AS pos
  FROM soma
)
SELECT r.fonte, r.ano, r.uf, r.pos, r.cid_cat, d.descricao, r.n, r.pct, d.observacao
FROM ranking r
LEFT JOIN dim_cid d ON d.cid_cat = r.cid_cat
WHERE r.pos <= 3;

SELECT * FROM gold_lideres WHERE pos = 1 ORDER BY fonte, uf, ano LIMIT 30

## 3.6 Governança: publicar sem expor pessoas

Os microdados do SIM e do SIH não têm nome nem CPF, mas a combinação **município × ano × causa rara × sexo × idade** pode apontar uma pessoa, sobretudo em municípios pequenos. Esses campos são os *quasi-identificadores* do TCC de Pereira. O nosso `gold` agrega por **UF**, e o risco cai muito. Mesmo assim, aplicamos a regra da oficina: **nenhuma tabela publicada tem célula com menos de *k* = 5 eventos**.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_fato_causa_publico AS
SELECT * FROM gold_fato_causa WHERE n >= 5;          -- k = 5

SELECT count(*)                                AS celulas,
       count_if(n < 5)                         AS celulas_n_menor_5,
       round(count_if(n < 5) / count(*), 3)    AS pct
FROM gold_fato_causa

---
### Trilhas — Etapa 3

- **Prático:** troque o filtro `IDENT = '1'` por nenhum filtro. Quantas internações a mais aparecem? A líder de alguma UF muda?
- **Criativo:** escolha um óbito fictício de 2005 em um município de 7 dígitos e conte a "biografia" dele, do `.dbc` até a `gold_lideres`: em que linha de SQL cada atributo dele foi transformado?
- **Abstrato:** `gold_fato_causa` guarda **todos** os capítulos, e o filtro do 3.4 é aplicado na consulta. Compare com filtrar já no silver. Que pergunta futura cada escolha permite ou impede? Relacione com *schema-on-read* × *schema-on-write*.

---
# 4. Analyze

## 4.1 A doença que mais mata, por UF e ano

Uma linha por UF e uma coluna por ano. Em SQL, o `PIVOT` precisa da **lista explícita** de valores que viram colunas: o domínio do eixo do tempo é uma decisão do modelo, não algo que a consulta descobre sozinha. Anos sem dado (no modo `rapido`, a maioria) ficam com colunas vazias.

In [ ]:
%sql
SELECT *
FROM (SELECT uf, ano, cid_cat FROM gold_lideres WHERE fonte = 'obito' AND pos = 1)
PIVOT (first(cid_cat) FOR ano IN (1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025))
ORDER BY uf

## 4.2 A doença que mais causa internação, por UF e ano

In [ ]:
%sql
-- 🧩 SUA VEZ (TODO 4.1) — a mesma tabela para as internações
SELECT '🧩 TODO 4.1: escreva aqui a consulta (adapte a de cima)' AS todo

## 4.3 Mapa de líderes

Uma linha por UF e uma coluna por ano. Cada célula mostra o código da doença líder. Os dados vêm de uma consulta SQL; o Python só desenha. As **três** doenças que mais lideram ganham cor e as demais ficam em cinza: com mais cores, o olho não distingue mais as categorias. O código escrito dentro da célula garante que a leitura não dependa só da cor.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CORES = ["#2a78d6", "#eb6834", "#1baf7a"]          # 3 cores categóricas validadas para leitura daltônica
OUTRA, FUNDO, TINTA, TINTA2 = "#e6e5e0", "#fcfcfb", "#0b0b0b", "#52514e"
ROTULO_CURTO = {"B34": "COVID-19 (codificada como B34.2; ver dim_cid.observacao)"}

def mapa_lideres(fonte, titulo):
    lid = spark.sql(f"SELECT uf, ano, cid_cat, descricao FROM gold_lideres "
                    f"WHERE fonte = '{fonte}' AND pos = 1").toPandas()
    grade = lid.pivot(index="uf", columns="ano", values="cid_cat").sort_index(ascending=False)
    frequentes = lid["cid_cat"].value_counts().index[:3].tolist()
    cor = {c: CORES[i] for i, c in enumerate(frequentes)}
    desc = lid.drop_duplicates("cid_cat").set_index("cid_cat")["descricao"].to_dict()

    fig, ax = plt.subplots(figsize=(max(9, 0.45 * grade.shape[1] + 2), 0.32 * grade.shape[0] + 1.6), facecolor=FUNDO)
    ax.set_facecolor(FUNDO)
    for i, uf in enumerate(grade.index):
        for j, ano in enumerate(grade.columns):
            c = grade.loc[uf, ano]
            if isinstance(c, str):
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=cor.get(c, OUTRA), edgecolor=FUNDO, linewidth=2))
                ax.text(j + 0.5, i + 0.5, c, ha="center", va="center", fontsize=6, color=TINTA)
    ax.set_xlim(0, grade.shape[1]); ax.set_ylim(0, grade.shape[0])
    ax.xaxis.tick_top()
    ax.set_xticks([j + 0.5 for j in range(grade.shape[1])], grade.columns, rotation=90, fontsize=7, color=TINTA2)
    ax.set_yticks([i + 0.5 for i in range(grade.shape[0])], grade.index, fontsize=7, color=TINTA2)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(titulo, loc="left", fontsize=11, color=TINTA, pad=28)
    leg = [Patch(facecolor=cor[c], label=f"{c} — {ROTULO_CURTO.get(c) or desc.get(c) or 's/ descrição'}") for c in frequentes]
    leg.append(Patch(facecolor=OUTRA, label="outras"))
    ax.legend(handles=leg, loc="upper left", bbox_to_anchor=(0, -0.02), ncol=1, frameon=False, fontsize=8, labelcolor=TINTA2)
    plt.tight_layout()
    plt.show()

mapa_lideres("obito", "Doença que mais mata, por UF de residência e ano (categoria CID-10)")

In [ ]:
# 🧩 SUA VEZ (TODO 4.2) — o mapa das internações
...

## 4.4 Da mais antiga à mais recente: quando a liderança mudou?

Para cada UF e fonte: a líder no **primeiro** ano, a líder no **último**, quantas vezes a liderança trocou de mãos e em quais anos a `B34` (COVID-19) liderou.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_trajetoria AS
WITH lider AS (
  SELECT fonte, uf, ano, cid_cat,
         LAG(cid_cat) OVER (PARTITION BY fonte, uf ORDER BY ano) AS anterior
  FROM gold_lideres
  WHERE pos = 1
)
SELECT fonte, uf,
       min(ano)                                                AS de,
       max(ano)                                                AS ate,
       min_by(cid_cat, ano)                                    AS lider_inicio,
       max_by(cid_cat, ano)                                    AS lider_fim,
       count_if(anterior IS NOT NULL AND anterior <> cid_cat)  AS trocas,
       array_sort(collect_list(CASE WHEN cid_cat = 'B34' THEN ano END)) AS anos_B34
FROM lider
GROUP BY fonte, uf;

SELECT * FROM gold_trajetoria ORDER BY fonte, uf

🧩 **SUA VEZ (TODO 4.3) — a resposta do desafio, em texto.** Com a tabela acima e os dois mapas, escreva até 6 frases:

1. Qual doença mais mata **na maioria das UFs** no início e no fim da série? Há exceções regionais?
2. O que aconteceu em 2020–2021? Em quantas UFs a `B34` liderou e por quanto tempo?
3. Qual doença mais causa internação? Ela é a mesma que mais mata? Por que faz sentido que não seja?

*(escreva aqui)*

## 4.5 O resultado é robusto?

Um resultado só vira **insight** se sobreviver a decisões de modelagem razoáveis. A consulta abaixo recalcula a líder sob outras escolhas e mede **em quantos pares (UF, ano) a líder continua a mesma**.

Cada cenário é uma **linha** da CTE `cenarios`: o nível da CID e a lista de capítulos excluídos. Um `CROSS JOIN` aplica todos os cenários de uma vez ao fato. Para testar outra decisão, acrescente uma linha.

In [ ]:
%sql
WITH seed AS (SELECT collect_list(capitulo) AS fora FROM capitulos_fora),
cenarios AS (
  SELECT 1 AS ordem, 'sem excluir capítulos (parto, mal definidas, causas externas entram)' AS cenario,
         'cid_cat' AS nivel, CAST(array() AS ARRAY<INT>) AS fora
  UNION ALL SELECT 2, 'excluindo só gravidez/parto e mal definidas', 'cid_cat', array(15, 18)
  UNION ALL SELECT 3, 'doença = grupo CID (ex.: I20–I25 isquêmicas)', 'grupo', fora FROM seed
  UNION ALL SELECT 4, 'doença = capítulo CID (ex.: IX circulatório)', 'capitulo', fora FROM seed
),
fato AS (
  SELECT g.fonte, g.ano, g.uf, g.n, g.capitulo, g.cid_cat, d.grupo
  FROM gold_fato_causa g LEFT JOIN dim_cid d ON d.cid_cat = g.cid_cat
),
alt AS (                          -- a líder de cada (fonte, UF, ano) em cada cenário
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY ordem, fonte, ano, uf ORDER BY n DESC, chave) AS pos
    FROM (SELECT c.ordem, c.cenario, c.nivel, f.fonte, f.ano, f.uf,
                 CASE c.nivel WHEN 'grupo'    THEN f.grupo
                              WHEN 'capitulo' THEN CAST(f.capitulo AS STRING)
                              ELSE f.cid_cat END AS chave,
                 sum(f.n) AS n
          FROM fato f CROSS JOIN cenarios c
          WHERE NOT array_contains(c.fora, coalesce(f.capitulo, -1))
          GROUP BY 1, 2, 3, 4, 5, 6, 7))
  WHERE pos = 1
),
comparacao AS (                   -- a líder do seed "subida" para o nível do cenário
  SELECT a.ordem, a.cenario, a.fonte,
         CAST(a.chave = CASE a.nivel WHEN 'grupo'    THEN d.grupo
                                     WHEN 'capitulo' THEN CAST(d.capitulo AS STRING)
                                     ELSE b.cid_cat END AS INT) AS mesma
  FROM alt a
  JOIN gold_lideres b ON b.pos = 1 AND b.fonte = a.fonte AND b.ano = a.ano AND b.uf = a.uf
  LEFT JOIN dim_cid d ON d.cid_cat = b.cid_cat
)
SELECT cenario,
       round(avg(CASE WHEN fonte = 'obito'      THEN mesma END), 3) AS `óbitos: mesma líder`,
       round(avg(CASE WHEN fonte = 'internacao' THEN mesma END), 3) AS `internações: mesma líder`
FROM comparacao
GROUP BY ordem, cenario
ORDER BY ordem

> **Como ler.** 1,0 = a decisão não muda nada; 0,3 = a resposta depende quase toda da decisão. No cenário de **grupo** e de **capítulo**, comparamos a líder do seed "subida" para o mesmo nível: se ela continua líder, há concordância. Uma concordância baixa **não é erro**: é a prova de que a sua decisão de modelagem precisa aparecer no relatório.
>
> Há uma quinta decisão que esta célula não testa: o **mês** do SIH. No modo `completo` usamos 4 meses por ano (um por trimestre). Doenças respiratórias são sazonais. Se só janeiro fosse usado, a pneumonia de inverno sumiria. Teste isso na trilha prática abaixo.

## 4.6 (Opcional) Um novo atributo na `dim_cid` com o Jev

A regra do 3.4 classifica por **capítulo**, de forma determinística. Ela não sabe, por exemplo, que `B34` em 2021 é COVID-19, nem que `Z51` são sessões de quimioterapia. Este é o padrão da **A2**: do texto (a descrição da doença) nasce um **atributo tipado em uma dimensão existente**.

Há dois caminhos para o mesmo **modelo de decisão** (um modelo que não gera texto: devolve julgamentos tipados com probabilidades). Os dois são **opcionais** e aparecem aqui de forma superficial; faça um, outro ou nenhum:

| | 4.6 — TypeSafe (Jev) | 4.7 — Databricks `ai_decide` |
|---|---|---|
| Onde roda | Python, via `typesafe-sdk` (aqui, no Codespaces ou em qualquer lugar) | SQL, dentro do Databricks |
| Chave | sua chave TypeSafe, num *secret scope* | nenhuma: usa a governança do workspace |
| Perguntas | `Choice`, `Score`, `Noul` | o mesmo JSON: `type` = `choice`, `score`, `noul` |
| Status | API estável; pinar a versão do modelo | **Beta**, só em algumas regiões; pode não existir no seu workspace |

Para cada categoria que já apareceu entre as 3 primeiras (`gold_lideres`), o Jev responde a **duas perguntas na mesma requisição** (*fan-out*):

| id | tipo | pergunta |
|---|---|---|
| `natureza` | **Choice** | Que tipo de condição é `cid.descricao`? |
| `evitavel` | **Noul** | Mortes e internações por essa condição são em grande parte evitáveis com atenção primária, prevenção ou diagnóstico precoce? |

**Antes de rodar**, rotule à mão a `natureza` de 10 categorias. O Jev é instrumento de medição, não oráculo: confiança baixa é conteúdo de aula, não erro.

### Antes de rodar: guardar a chave da TypeSafe num *secret scope*

A chave da API **nunca** vai numa célula. O notebook a lê com `dbutils.secrets.get("typesafe", "api_key")`, e o Databricks mostra `[REDACTED]` se ela aparecer na saída. Mesmo apagada depois, uma chave colada numa célula fica no **histórico de revisões** do notebook. Sem a chave, a seção é pulada com um aviso e o resto do notebook roda normalmente.

Os comandos `databricks secrets …` são da **CLI do Databricks**. Eles rodam num **terminal** (Codespaces ou seu computador), não numa célula do notebook:

1. **Crie a chave** em https://console.typesafe.ai/ e copie-a.
2. **Instale a CLI** (versão 0.205 ou mais nova):
   - Linux, macOS ou Codespaces: `curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh` (se reclamar de permissão em `/usr/local/bin`, rode de novo com `sudo`)
   - macOS com Homebrew: `brew install databricks/tap/databricks`
   - Windows: `winget install Databricks.DatabricksCLI` (depois, reabra o terminal)
   - Confira com `databricks -v`.
3. **Faça login no seu workspace.** A URL é o início do endereço do Databricks no navegador, por exemplo `https://dbc-1234abcd-5678.cloud.databricks.com`:
   ```bash
   databricks auth login --host https://<seu-workspace>.cloud.databricks.com
   ```
   O navegador abre para você entrar com a sua conta. Aceite o nome de perfil sugerido.
4. **Crie o scope e guarde a chave:**
   ```bash
   databricks secrets create-scope typesafe
   databricks secrets put-secret typesafe api_key
   ```
   O segundo comando **pede o valor** no terminal: cole a chave e tecle Enter. Ela não aparece na tela e não fica no histórico do shell.
5. **Confira:** `databricks secrets list-secrets typesafe` deve listar `api_key`. A lista mostra só o nome, nunca o valor.

**Voltou ao notebook depois de configurar a chave?** Se a sessão ficou ociosa, o Python pode ter reiniciado, e as definições das seções anteriores (`T`, `dim_cid`…) se perderam. As células abaixo avisam quando isso acontece. Nesse caso, use **Run → Run all above**: o notebook é idempotente, então nada é baixado de novo.

Se o seu workspace não permitir criar *secret scopes*, pule a 4.6. Ela é opcional, e a 4.7 (`ai_decide`) não precisa de chave.

In [ ]:
# As perguntas do Jev (a 4.7 usa o mesmo texto, em SQL).
if "T" not in globals():   # o Python reiniciou (sessão ociosa expirou, compute reconectado...)
    raise RuntimeError("As definições das seções anteriores se perderam porque o Python reiniciou. "
                       "Clique nesta célula e use Run → Run all above (nada é baixado de novo); depois rode-a outra vez.")
NATUREZA_INSTR = 'Que tipo de condição de saúde descreve a categoria `cid.codigo` da CID-10, `cid.descricao`?'
NATUREZA_CRITERIA = {
    "cronica": "Doença crônica não transmissível: cardiovascular, câncer, diabetes, respiratória crônica, renal crônica",
    "infecciosa": "Doença infecciosa ou parasitária, aguda ou crônica, incluindo pneumonias e infecções virais",
    "gravidez_parto": "Gravidez, parto, puerpério ou condição do recém-nascido",
    "externa": "Lesão, envenenamento, acidente, agressão ou outra causa externa",
    "mal_definida": "Sintoma, sinal ou causa mal definida, sem diagnóstico de doença",
    "contato_servico": "Procedimento, cuidado de saúde ou motivo de contato com o serviço, não uma doença"
}
EVITAVEL_INSTR = 'Mortes e internações por `cid.descricao` são em grande parte evitáveis no Brasil com atenção primária, prevenção ou diagnóstico precoce?'

In [ ]:
if "T" not in globals():   # o Python reiniciou (sessão ociosa expirou, compute reconectado...)
    raise RuntimeError("As definições das seções anteriores se perderam porque o Python reiniciou. "
                       "Clique nesta célula e use Run → Run all above (nada é baixado de novo); depois rode-a outra vez.")
from typesafe_sdk import TypeSafeClient, Choice, Noul

def chave_typesafe():
    try:
        return dbutils.secrets.get("typesafe", "api_key")
    except Exception:
        return os.environ.get("TYPESAFE_API_KEY")

PERGUNTAS = {
    "natureza": Choice(instructions=NATUREZA_INSTR, criteria=NATUREZA_CRITERIA),
    "evitavel": Noul(instructions=EVITAVEL_INSTR),
}

categorias = spark.sql("SELECT DISTINCT d.cid_cat, d.descricao, d.capitulo, d.observacao "
                       "FROM gold_lideres l JOIN dim_cid d ON d.cid_cat = l.cid_cat").toPandas()
ESQUEMA_JEV = ("cid_cat STRING, natureza STRING, natureza_conf DOUBLE, natureza_probs STRING, "
               "evitavel DOUBLE, modelo STRING")
chave = chave_typesafe()
if not chave:
    print("⚠️  Sem chave da TypeSafe (secret scope `typesafe`, chave `api_key`): a seção 4.6 é opcional e foi pulada.")
    if not spark.catalog.tableExists(T("silver_cid_jev")):     # vazia: o SQL abaixo roda mesmo assim
        spark.createDataFrame([], ESQUEMA_JEV).write.saveAsTable(T("silver_cid_jev"))
else:
    resultados = []
    with TypeSafeClient(api_key=chave, model="jev-latest") as client:   # em produção: pinar a versão
        for c in categorias.itertuples():
            state = {"cid": {"codigo": c.cid_cat, "descricao": c.descricao,
                             **({"observacao": c.observacao} if c.observacao else {})}}
            r = client.system_one(state=state, questions=PERGUNTAS)
            n, e = r.choices["natureza"], r.nouls["evitavel"]
            resultados.append({"cid_cat": c.cid_cat, "natureza": n.choice, "natureza_conf": n.confidence,
                               "natureza_probs": json.dumps(n.probabilities), "evitavel": e.noul, "modelo": r.model})
    spark.createDataFrame(pd.DataFrame(resultados), ESQUEMA_JEV).write.mode("overwrite") \
         .option("overwriteSchema", True).saveAsTable(T("silver_cid_jev"))

### Composição em SQL: o novo atributo e a comparação com a regra

- `natureza` entra na `dim_cid` só com confiança ≥ 0,6; abaixo disso, `indefinido` (revisão humana).
- `evitavel` vira `sim` (≥ 0,7), `não` (≤ 0,3) ou `incerto`.
- **Divergência com a regra:** o Jev diz `cronica` ou `infecciosa`, mas o capítulo está em `capitulos_fora`, ou o contrário. Cada divergência é um caso para discutir.

O novo atributo entra na dimensão **existente** com `ALTER TABLE … ADD COLUMNS` e `MERGE INTO`, sem recriar a tabela. Esta é a operação pedida na A2. Como sobrescrevemos o valor, é um **SCD tipo 1**.

> **DDL não é idempotente.** Rodar esta célula duas vezes falha no `ALTER TABLE` (a coluna já existe). Para repetir, rode antes a célula da `dim_cid` (3.3), que recria a dimensão. Sem chave da TypeSafe, `silver_cid_jev` está vazia: as colunas são criadas e ficam nulas.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW atributos_jev AS
SELECT cid_cat,
       CASE WHEN natureza_conf >= 0.6 THEN natureza ELSE 'indefinido' END AS natureza,
       CASE WHEN evitavel >= 0.7 THEN 'sim' WHEN evitavel <= 0.3 THEN 'não' ELSE 'incerto' END AS evitavel
FROM silver_cid_jev;

ALTER TABLE dim_cid ADD COLUMNS (natureza STRING, evitavel STRING);

MERGE INTO dim_cid d
USING atributos_jev j ON d.cid_cat = j.cid_cat
WHEN MATCHED THEN UPDATE SET d.natureza = j.natureza, d.evitavel = j.evitavel;

SELECT d.cid_cat, d.descricao, d.capitulo, d.natureza, d.evitavel,
       (f.capitulo IS NOT NULL) <> (d.natureza NOT IN ('cronica', 'infecciosa', 'indefinido')) AS divergencia
FROM dim_cid d
LEFT JOIN capitulos_fora f ON f.capitulo = d.capitulo
WHERE d.natureza IS NOT NULL
ORDER BY divergencia DESC, d.cid_cat

## 4.7 (Opcional) O mesmo julgamento em SQL com `ai_decide`

O Databricks tem uma função SQL nativa, **`ai_decide(state, questions)`**, movida por um modelo de decisão da TypeSafe. O JSON de `questions` é o **mesmo formato** do System One: `type`, `instructions` e `criteria`. Não há chave para guardar, e o dado não sai do workspace.

- `state`: aqui, um JSON por linha montado com `to_json(named_struct(...))`.
- `questions`: o mesmo texto das perguntas da 4.6, escrito como constante JSON.
- Resposta: um `VARIANT` com `response.answers.<id>`. Atenção aos nomes: no `noul`, o valor vem em **`probability`**; no `choice`, em `choice`, `probabilities` e `confidence`.

A função está em **Beta** e só existe em algumas regiões. Se o seu workspace não a tiver, a célula dá erro (`UNRESOLVED_ROUTINE`): é a última célula do roteiro, então nada depois dela é afetado. Basta ler o SQL.

In [ ]:
%sql
USE workspace.oficina_sus;

WITH decisoes AS (
  SELECT cid_cat, descricao,
         ai_decide(to_json(named_struct('cid', named_struct('codigo', cid_cat, 'descricao', descricao,
                                                            'observacao', observacao))),
                   '{"natureza": {"type": "choice", "instructions": "Que tipo de condição de saúde descreve a categoria `cid.codigo` da CID-10, `cid.descricao`?", "criteria": {"cronica": "Doença crônica não transmissível: cardiovascular, câncer, diabetes, respiratória crônica, renal crônica", "infecciosa": "Doença infecciosa ou parasitária, aguda ou crônica, incluindo pneumonias e infecções virais", "gravidez_parto": "Gravidez, parto, puerpério ou condição do recém-nascido", "externa": "Lesão, envenenamento, acidente, agressão ou outra causa externa", "mal_definida": "Sintoma, sinal ou causa mal definida, sem diagnóstico de doença", "contato_servico": "Procedimento, cuidado de saúde ou motivo de contato com o serviço, não uma doença"}}, "evitavel": {"type": "noul", "instructions": "Mortes e internações por `cid.descricao` são em grande parte evitáveis no Brasil com atenção primária, prevenção ou diagnóstico precoce?"}}') AS d
  FROM dim_cid
  WHERE cid_cat IN (SELECT cid_cat FROM gold_lideres WHERE pos = 1)
  LIMIT 10
),
lidas AS (
  SELECT cid_cat, descricao,
         d:response.answers.natureza.choice::string                AS natureza_sql,
         round(d:response.answers.natureza.confidence::double, 2)  AS confianca_sql,
         round(d:response.answers.evitavel.probability::double, 2) AS evitavel_sql,
         d:error_message::string                                   AS erro
  FROM decisoes
)
SELECT l.*, j.natureza AS natureza_jev, round(j.evitavel, 2) AS evitavel_jev   -- nulos se não fez a 4.6
FROM lidas l
LEFT JOIN silver_cid_jev j ON j.cid_cat = l.cid_cat

**Para discutir:** o `ai_decide` e o Jev do 4.6 concordam? Onde divergem, a confiança é baixa? Qual dos dois caminhos você usaria **dentro de um pipeline agendado** do Databricks, e qual fora dele? O que muda em **governança** (onde o dado trafega, quem guarda a chave)?

---
### Trilhas — Etapa 4

- **Prático:** recalcule as internações de 2019 usando só `mes = 1` e depois só `mes = 7` (a coluna `mes` está em `silver_internacoes`). A líder muda em quantas UFs? O que isso diz sobre amostrar meses?
- **Criativo:** escreva a **manchete** do seu achado (até 12 palavras) e o lide (2 frases), sustentados pelo mapa da 4.3. Exemplo de tom: *"Por dois anos, um vírus tirou o infarto do topo em quase todo o país"*.
- **Abstrato:** a resposta muda com o **nível** da CID (4.5). Prove com um exemplo numérico pequeno (3 categorias, 2 grupos) que a categoria líder **pode não pertencer** ao grupo líder. Que nível você defenderia para um gestor? E se a pergunta fosse sobre CID-9 (1979–1995): o que precisaria existir para estender a série?

---
# 5. Entrega (individual)

Exporte este notebook executado no modo **`completo`** (**File → Export → IPython Notebook**) e entregue no D2L, junto com um texto de até 1 página contendo:

1. **A resposta do desafio** (TODO 4.3), com os dois mapas da seção 4.3.
2. **Dicionário de dados** do SIH (TODO 1.1) e as **4 decisões de modelagem** que você tomou: nível da CID, capítulos excluídos, UF de residência × de atendimento, meses do SIH.
3. **Robustez**: a tabela da 4.5 e uma frase interpretando-a.
4. **Governança**: o percentual de células com *n* < *k* (3.6) e como você evitou publicá-las.
5. **Trilhas**: quais atividades você fez em cada etapa.
6. (Opcional) A tabela de divergências regra × Jev (4.6) e um caso comentado; se rodou o 4.7, uma frase comparando Jev × `ai_decide`.

### Checklist de autoavaliação

- [ ] As tabelas `bronze_*`, `silver_*`, `dim_cid` e `gold_*` existem no schema `oficina_sus`.
- [ ] Rodar o notebook duas vezes não baixa nem duplica nada (idempotência).
- [ ] Não li nenhum arquivo `DOBR…` e não confundi o prefixo de arquivo `RJ` do SIH com o estado.
- [ ] Contei só AIH com `IDENT = 1`.
- [ ] Sei explicar por que a série começa em 1996 (SIM) e 2008 (SIH), e por que 2026 ficou de fora.
- [ ] Sei explicar por que `B34` é COVID-19 nesta base.
- [ ] Nenhuma chave de API aparece no notebook.